In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd

from bs4 import BeautifulSoup

from time import sleep

from datetime import datetime

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

import datetime

import os

from selenium.webdriver.chrome.service import Service as ChromeService

import re


# %%

In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'BA FBIH'

print(f"Running{regulatorName} Web Scraping Tool v.1.2")


now=datetime.datetime.now()

filename= 'BA FBIH Data {}.xlsx'.format(str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') 



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


RunningBA FBIH Web Scraping Tool v.1.2


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

In [4]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def find_zip_code(string):
    match = re.search(r'\d{5}', string)
    if match:
        return match.group()
    else:
        return None

In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------


regdict={'BA FBIH 1': 'https://www.fba.ba/eng/banks-in-federation-of-bosnia-and-herzegovina', 
         'BA FBIH 2': 'https://www.fba.ba/eng/mcos-in-the-federation-of-bih',
         'BA FBIH 3': 'https://www.fba.ba/eng/leasing-companies-in-the-federation-of-bih',
         'BA FBIH 4': 'https://www.fba.ba/eng/banks-seated-in-republika-srpska-which-have-the-organizational-units-in-the-federation-of-bih',
         'BA FBIH 5': 'https://www.fba.ba/eng/mcos-seated-in-republika-srpska-which-have-the-organizational-units-in-the-federation-of-bih'
         }


Typology={'BA FBIH 1': 'Banks in Federation of Bosnia and Herzegovina', 
         'BA FBIH 2': 'MCOs in the Federation of BiH',
         'BA FBIH 3': 'Leasing Companies in the Federation of BiH',
         'BA FBIH 4': 'Banks seated in Republika Srpska which have the organizational units in the Federation of BiH',
         'BA FBIH 5': 'MCOs seated in Republika Srpska which have the organizational units in the Federation of BiH'
         }
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')
		  

In [6]:
# %%
#------------------------------------------------ Begin_Main ----------------------------------------


for reg in regdict:
	driver.delete_all_cookies()
	inner_links = []
	print('Working with {}.'.format(reg))
	driver.get(regdict[reg])
	sleep(3)
	soup=BeautifulSoup(driver.page_source, "html.parser")
	
	
	try:
		serverError = soup.find('div',{'class':'code'})== 500 or 'Server Error' in soup.find('div',{'class':'message'}).text
	except:
		serverError = 0
  
	if serverError:
		driver.refresh()
		sleep(2)
		soup=BeautifulSoup(driver.page_source, "html.parser")
		sleep(2)
	data = soup.find("div", {"id":"category-list"}).find_all('div',{'class':'item2'})
	inner_links = []
	for info in data:
		href = info.find('a')['href']
		inner_links.append('https://www.fba.ba'+href)
  
	for inner_link in inner_links:
		driver.get(inner_link)
		sleep(2)
		soup2 = BeautifulSoup(driver.page_source, "html.parser")
		sleep(1)
		regulateddate = soup2.find('article').find('div',{'class':'meta'}).text.split('|')[-1].strip()
		sqldict['RegulationDate'].append(regulateddate)
		title = soup2.find('article').find('h2').text
		sqldict['Name'].append(title)
		sqldict['ListProcessDate'].append(processdate)
		sleep(1)

		if reg == 'BA FBIH 1':
			list_info = soup2.find('article').find('p').text.split('\n')
			address = list_info[2].split(':')[-1]
			address = str(address).strip()
			zip_code = find_zip_code(address)
			
			if zip_code:		
				city = address[address.find(zip_code)+6:]
				#print(city)
				sqldict['Zip'].append(zip_code)
				sqldict['City'].append(city)
			else:
				sqldict['Zip'].append('')
				sqldict['City'].append('')
	
			phone = list_info[3].split(':')[-1]
			sqldict['Phone'].append(phone)

			fax = list_info[4].split(':')[-1]
			sqldict['Fax'].append(fax)
	
			email = list_info[5].split(':')[-1]
			sqldict['Email'].append(email)
	
			web = list_info[6].split(':')[-1]
			sqldict['Website'].append(web)
		
		else:
			# try:
				text_info = soup2.find('article').text.split('\n')
				for i,data in enumerate(text_info):
					if ':' in data:
						#print(i,data)
						if data.find('Adresa')!=-1:
							address = str(data.split(':')[-1]).strip()
							zip_code = find_zip_code(address)
							if zip_code:		
								city = address[address.find(zip_code)+6:]
								sqldict['Zip'].append(zip_code)
								sqldict['City'].append(city)
							else:
								sqldict['Zip'].append('')
								sqldict['City'].append('')
						elif data.find('Telefon')!=-1:
							phone = data.split(':')[-1]
							sqldict['Phone'].append(phone)
						elif data.find('Fax')!=-1:
							fax = data.split(':')[-1]
							sqldict['Fax'].append(fax)
						elif data.find('E-mail')!=-1:
							email = data.split(':')[-1]
							sqldict['Email'].append(email)
						elif data.find('Web')!=-1:
							web = data.split(':')[-1]
							sqldict['Website'].append(web)
	
		sqldict['RegCtry'].append(reg.split()[0])
		sqldict['RegCode'].append(reg.split()[1])
		sqldict['ListCode'].append(reg.split()[2])
		sqldict['ListName'].append(Typology[reg])
		sqldict['RegulationType'].append('Regulated')
		sqldict = bourange_same_length_array(sqldict) 

	
sqldict = bourange_same_length_array(sqldict) 

# %%

Working with BA FBIH 1.
Working with BA FBIH 2.


IndexError: list index out of range

In [8]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\9\ipykernel_25876\3068940208.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [9]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 42 values.
Key 'priority' has 42 values.
Key 'ListLabel' has 42 values.
Key 'Typology' has 42 values.
Key 'EntryType' has 42 values.
Key 'Name' has 42 values.
Key 'InternalID_1' has 42 values.
Key 'InternalID_1_type' has 42 values.
Key 'InternalID_2' has 42 values.
Key 'InternalID_2_type' has 42 values.
Key 'InternalID_3' has 42 values.
Key 'InternalID_3_type' has 42 values.
Key 'CoType' has 42 values.
Key 'License_Type' has 42 values.
Key 'Address_1' has 42 values.
Key 'Address_2' has 42 values.
Key 'City' has 42 values.
Key 'Zip' has 42 values.
Key 'Cntry' has 42 values.
Key 'Phone' has 42 values.
Key 'Fax' has 42 values.
Key 'Website' has 42 values.
Key 'Email' has 42 values.
Key 'RegulationType' has 42 values.
Key 'RegulationTypeCode' has 42 values.
Key 'RegulationDate' has 42 values.
Key 'CancellationDate' has 42 values.
Key 'RegCtry' has 42 values.
Key 'RegCode' has 42 values.
Key 'ListCode' has 42 values.
Key 'ListLanguage' has 42 values.
Key 'ListValidityDate' h

In [10]:

df.to_csv('list 1 - 5.csv')